# Selecionar grupo controle facilities

In [12]:
import pandas as pd
import math
import pandas as pd
import numpy as np

# Datacenters

In [2]:
datacenter = pd.read_csv('datacenter_filtrado.csv', sep=';', encoding="utf-8-sig")
map_names = {'Sao Joao de Meriti': 'São João de Meriti'}
datacenter['cidade'] = datacenter['cidade'].map(map_names).fillna(datacenter['cidade'])

In [40]:
datacenter

,id_datacenter,nome_datacenter,endereco,cidade,estado,latitude,longitude,tags,mw_construido,whitespace_construido_m,ano_operacional,tipo_construcao
0,dc_1e8c6816f0,Ascenty - Jundiai JDI2,"Av. Beirute, 863",Jundiaí,Sao Paulo,-23.192467,-46.972989,Carrier Neutral,13.0,8993.0,2019,Purpose-built
1,dc_211c584638,Ascenty - Sumare SUM2,"Rodovia Anhanguera s/n, Parque das Industrias ...",Sumaré,Campinas,-22.813862,-47.213363,Carrier Neutral,20.0,NaN,2019,NaN
2,dc_23c0e2a666,Ascenty - Hortolandia HTL2,"Rua Papa João Paulo II, 4",Hortolândia,Campinas,-22.894969,-47.179440,Carrier Neutral,7.0,2991.0,2019,Retrofitted
3,dc_2b7c5d95d1,Ascenty - Hortolandia HTL4,"Rua Papa João Paulo II, 4",Hortolândia,Campinas,-22.894858,-47.178904,Carrier Neutral,3.0,1997.0,2021,Retrofitted
4,dc_3357273eaa,Ascenty - Hortolandia HTL3,"Rua Papa João Paulo II, 4",Hortolândia,Campinas,-22.894209,-47.179702,Carrier Neutral,3.0,1997.0,2019,Retrofitted
5,dc_363da00599,Ascenty - Sao Paulo SP3,"Avenida Roberto Pinto Sobrinho, 350",Osasco,Sao Paulo,-23.491232,-46.775931,Carrier Neutral,4.0,3995.0,2020,Retrofitted
6,dc_541dfaf17f,Ascenty - Paulinia PLN1,R. Sebastião Cardoso 350,Paulínia,Campinas,-22.797079,-47.135230,Carrier Neutral,13.0,NaN,2019,Purpose-built
7,dc_6fba998120,Ascenty - Hortolandia HTL5,"Rua Papa João Paulo II, 4",Hortolândia,Campinas,-22.895892,-47.180192,Carrier Neutral,30.0,22984.0,2022,Purpose-built
8,dc_708ead2d07,Scala Data Centers SPOAPA01,Av. Pernambuco 1124,Porto Alegre,Porto Alegre,-30.002805,-51.197962,Hyperscale-Ready,8.4,NaN,2023,Purpose-built
9,dc_c81b687e80,Scala Data Centers SGIGSM01,Rod. Pres. Dutra 4300,São João de Meriti,Rio de Janeiro,-22.796438,-43.356317,Hyperscale-Ready,13.2,NaN,2023,Purpose-built


# Cidades similares

In [3]:
df_similar = pd.read_csv('municipios_com_cidades_similares.csv')
df_similar = df_similar[['nome_municipio', 'id_municipio', 'id_municipio_similar_1']]

# Dados lat, long

In [6]:
municipios = pd.read_csv('features_ibge_municipios_2016_2026.csv', sep=',', encoding="utf-8-sig")
municipios = municipios[['nome_municipio', 'id_municipio', 'latitude', 'longitude']].drop_duplicates().reset_index(drop=True)

In [7]:
municipios.head()

,nome_municipio,id_municipio,latitude,longitude
0,Brasília,5300108,-15.794087,-47.887905
1,Abadia de Goiás,5200050,-16.758812,-49.440548
2,Abadiânia,5200100,-16.203182,-48.706898
3,Acreúna,5200134,-17.398156,-50.374973
4,Adelândia,5200159,-16.414003,-50.165522


# Merge data

In [41]:
df = datacenter.merge(df_similar, left_on='cidade', right_on='nome_municipio', how='left')
df = df.merge(municipios, left_on='id_municipio', right_on='id_municipio', how='left', suffixes=('', '_origem'))
df = df.merge(municipios, left_on='id_municipio_similar_1', right_on='id_municipio', how='left', suffixes=('', '_similar'))

In [42]:
df.rename(columns={'latitude': 'latitude_datacenter', 'longitude': 'longitude_datacenter',
                   'latitude_origem': 'latitude_municipio', 'longitude_origem': 'longitude_municipio'
                   }, inplace=True)

In [43]:
coordenadas = df[['id_datacenter', 'latitude_datacenter', 'longitude_datacenter', 'latitude_municipio', 'longitude_municipio', 'latitude_similar', 'longitude_similar']].copy()

# Calcula distancias e controle

In [44]:
import math
import pandas as pd

def calcular_distancia_e_inclinacao(lat_cidade, lon_cidade, lat_dc, lon_dc):
    """
    Calcula distância (km) e inclinação/azimute (0-360°) entre dois pontos.
    """
    # Tratamento para valores ausentes
    if pd.isna(lat_cidade) or pd.isna(lon_cidade) or pd.isna(lat_dc) or pd.isna(lon_dc):
        return None, None

    # Converter de graus para radianos
    phi1, phi2 = math.radians(lat_cidade), math.radians(lat_dc)
    d_phi = math.radians(lat_dc - lat_cidade)
    d_lam = math.radians(lon_dc - lon_cidade)

    # 1. Distância (Haversine)
    a = math.sin(d_phi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(d_lam / 2)**2
    distancia_km = 6371.0 * (2 * math.atan2(math.sqrt(a), math.sqrt(1 - a)))

    # 2. Inclinação / Azimute (Norte = 0°)
    y = math.sin(d_lam) * math.cos(phi2)
    x = math.cos(phi1) * math.sin(phi2) - math.sin(phi1) * math.cos(phi2) * math.cos(d_lam)
    inclinacao_graus = (math.degrees(math.atan2(y, x)) + 360) % 360

    return round(distancia_km, 2), round(inclinacao_graus, 2)

In [45]:
coordenadas[['distancia_km', 'inclinacao_graus']] = coordenadas.apply(
    lambda row: pd.Series(
        calcular_distancia_e_inclinacao(
            row['latitude_municipio'], 
            row['longitude_municipio'], 
            row['latitude_datacenter'], 
            row['longitude_datacenter']
        )
    ), 
    axis=1
)

In [46]:
coordenadas

,id_datacenter,latitude_datacenter,longitude_datacenter,latitude_municipio,longitude_municipio,latitude_similar,longitude_similar,distancia_km,inclinacao_graus
0,dc_1e8c6816f0,-23.192467,-46.972989,-23.185054,-46.884119,-23.306157,-47.132732,9.12,264.80
1,dc_211c584638,-22.813862,-47.213363,-22.820902,-47.267176,-23.088945,-47.218010,5.57,81.93
2,dc_23c0e2a666,-22.894969,-47.179440,-22.858395,-47.221097,-23.303938,-45.965780,5.90,133.63
3,dc_2b7c5d95d1,-22.894858,-47.178904,-22.858395,-47.221097,-23.303938,-45.965780,5.93,133.18
4,dc_3357273eaa,-22.894209,-47.179702,-22.858395,-47.221097,-23.303938,-45.965780,5.82,133.21
5,dc_363da00599,-23.491232,-46.775931,-23.531148,-46.791619,-23.185054,-46.884119,4.72,19.82
6,dc_541dfaf17f,-22.797079,-47.135230,-22.759795,-47.154120,-23.085057,-46.950508,4.58,154.97
7,dc_6fba998120,-22.895892,-47.180192,-22.858395,-47.221097,-23.303938,-45.965780,5.91,134.86
8,dc_708ead2d07,-30.002805,-51.197962,-30.031430,-51.229989,-25.426337,-49.273028,4.43,44.10
9,dc_c81b687e80,-22.796438,-43.356317,-22.802581,-43.372150,-23.521149,-46.835509,1.76,67.18


In [34]:
def calcular_6_pontos_em_volta(lat_origem, lon_origem, distancia_km, inclinacao_inicial=0.0):
    """
    Gera 6 pontos em círculo ao redor da origem (intervalos de 60°).
    Retorna uma lista de dicionários com número do ponto, azimute, lat e lon.
    """
    if pd.isna(lat_origem) or pd.isna(lon_origem) or pd.isna(distancia_km):
        return []

    RAIO_TERRA_KM = 6371.0
    pontos = []

    lat1 = math.radians(lat_origem)
    lon1 = math.radians(lon_origem)
    d_r = distancia_km / RAIO_TERRA_KM

    # 6 pontos espaçados a cada 60° (360 / 6 = 60)
    for i in range(6):
        azimute_graus = (inclinacao_inicial + (i * 60.0)) % 360.0
        azimute = math.radians(azimute_graus)

        # Cálculo da nova Latitude
        lat2 = math.asin(
            math.sin(lat1) * math.cos(d_r) + 
            math.cos(lat1) * math.sin(d_r) * math.cos(azimute)
        )

        # Cálculo da nova Longitude
        lon2 = lon1 + math.atan2(
            math.sin(azimute) * math.sin(d_r) * math.cos(lat1),
            math.cos(d_r) - math.sin(lat1) * math.sin(lat2)
        )

        pontos.append({
            'ponto_num': i + 1,
            'ponto_inclinacao_graus': round(azimute_graus, 2),
            'ponto_latitude': round(math.degrees(lat2), 6),
            'ponto_longitude': round(math.degrees(lon2), 6)
        })

    return pontos

In [47]:
novas_linhas = []

for _, row in coordenadas.iterrows():
    # Calcula os 6 pontos para o município da linha atual
    pontos_gerados = calcular_6_pontos_em_volta(
        lat_origem=row['latitude_similar'],
        lon_origem=row['longitude_similar'],
        distancia_km=row['distancia_km'],
        inclinacao_inicial=0.0  # Ponto 1 no Norte (0°). Se quiser usar a inclinação do DC, mude para: row['inclinacao_graus']
    )
    
    # Para cada um dos 6 pontos, combina os dados originais + novos pontos
    for ponto in pontos_gerados:
        dados_linha = row.to_dict()
        dados_linha.update(ponto)
        novas_linhas.append(dados_linha)

In [48]:
df_expandido = pd.DataFrame(novas_linhas)

In [49]:
df_expandido.head(12)

,id_datacenter,latitude_datacenter,longitude_datacenter,latitude_municipio,longitude_municipio,latitude_similar,longitude_similar,distancia_km,inclinacao_graus,ponto_num,ponto_inclinacao_graus,ponto_latitude,ponto_longitude
0,dc_1e8c6816f0,-23.192467,-46.972989,-23.185054,-46.884119,-23.306157,-47.132732,9.12,264.80,1,0.0,-23.224139,-47.132732
1,dc_1e8c6816f0,-23.192467,-46.972989,-23.185054,-46.884119,-23.306157,-47.132732,9.12,264.80,2,60.0,-23.265129,-47.055415
2,dc_1e8c6816f0,-23.192467,-46.972989,-23.185054,-46.884119,-23.306157,-47.132732,9.12,264.80,3,120.0,-23.347147,-47.055368
3,dc_1e8c6816f0,-23.192467,-46.972989,-23.185054,-46.884119,-23.306157,-47.132732,9.12,264.80,4,180.0,-23.388175,-47.132732
4,dc_1e8c6816f0,-23.192467,-46.972989,-23.185054,-46.884119,-23.306157,-47.132732,9.12,264.80,5,240.0,-23.347147,-47.210096
5,dc_1e8c6816f0,-23.192467,-46.972989,-23.185054,-46.884119,-23.306157,-47.132732,9.12,264.80,6,300.0,-23.265129,-47.210049
6,dc_211c584638,-22.813862,-47.213363,-22.820902,-47.267176,-23.088945,-47.218010,5.57,81.93,1,0.0,-23.038853,-47.218010
7,dc_211c584638,-22.813862,-47.213363,-22.820902,-47.267176,-23.088945,-47.218010,5.57,81.93,2,60.0,-23.063892,-47.170860
8,dc_211c584638,-22.813862,-47.213363,-22.820902,-47.267176,-23.088945,-47.218010,5.57,81.93,3,120.0,-23.113984,-47.170843
9,dc_211c584638,-22.813862,-47.213363,-22.820902,-47.267176,-23.088945,-47.218010,5.57,81.93,4,180.0,-23.139037,-47.218010


In [51]:
df_expandido.to_csv('datacenter_expandido_6_pontos.csv', sep=';', index=False, encoding="utf-8-sig")

In [52]:
datacenter

,id_datacenter,nome_datacenter,endereco,cidade,estado,latitude,longitude,tags,mw_construido,whitespace_construido_m,ano_operacional,tipo_construcao
0,dc_1e8c6816f0,Ascenty - Jundiai JDI2,"Av. Beirute, 863",Jundiaí,Sao Paulo,-23.192467,-46.972989,Carrier Neutral,13.0,8993.0,2019,Purpose-built
1,dc_211c584638,Ascenty - Sumare SUM2,"Rodovia Anhanguera s/n, Parque das Industrias ...",Sumaré,Campinas,-22.813862,-47.213363,Carrier Neutral,20.0,NaN,2019,NaN
2,dc_23c0e2a666,Ascenty - Hortolandia HTL2,"Rua Papa João Paulo II, 4",Hortolândia,Campinas,-22.894969,-47.179440,Carrier Neutral,7.0,2991.0,2019,Retrofitted
3,dc_2b7c5d95d1,Ascenty - Hortolandia HTL4,"Rua Papa João Paulo II, 4",Hortolândia,Campinas,-22.894858,-47.178904,Carrier Neutral,3.0,1997.0,2021,Retrofitted
4,dc_3357273eaa,Ascenty - Hortolandia HTL3,"Rua Papa João Paulo II, 4",Hortolândia,Campinas,-22.894209,-47.179702,Carrier Neutral,3.0,1997.0,2019,Retrofitted
5,dc_363da00599,Ascenty - Sao Paulo SP3,"Avenida Roberto Pinto Sobrinho, 350",Osasco,Sao Paulo,-23.491232,-46.775931,Carrier Neutral,4.0,3995.0,2020,Retrofitted
6,dc_541dfaf17f,Ascenty - Paulinia PLN1,R. Sebastião Cardoso 350,Paulínia,Campinas,-22.797079,-47.135230,Carrier Neutral,13.0,NaN,2019,Purpose-built
7,dc_6fba998120,Ascenty - Hortolandia HTL5,"Rua Papa João Paulo II, 4",Hortolândia,Campinas,-22.895892,-47.180192,Carrier Neutral,30.0,22984.0,2022,Purpose-built
8,dc_708ead2d07,Scala Data Centers SPOAPA01,Av. Pernambuco 1124,Porto Alegre,Porto Alegre,-30.002805,-51.197962,Hyperscale-Ready,8.4,NaN,2023,Purpose-built
9,dc_c81b687e80,Scala Data Centers SGIGSM01,Rod. Pres. Dutra 4300,São João de Meriti,Rio de Janeiro,-22.796438,-43.356317,Hyperscale-Ready,13.2,NaN,2023,Purpose-built
